<a href="https://colab.research.google.com/github/pr3tzell/CSN/blob/main/Pandas_para_Ciencia_de_Datos_II_Proyecto_I_%E2%80%93_Parte_III_(Core)Javiera_Correa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import io
import pandas as pd
from google.colab import files
from IPython.display import display


# --------------------------------------------------
# 1. CARGAR Y REVISAR LOS DATOS
# --------------------------------------------------

print("Selecciona el archivo CSV utilizado en la Parte II.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Selecciona un solo archivo CSV.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))
datos.columns = datos.columns.str.strip()

# Renombrar columnas del inglés al español si existen en el archivo subido
renombrar_columnas = {
    "Product Category": "Producto",
    "Customer ID": "Tienda",
    "Quantity": "Ventas",
    "Price per Unit": "Precio",
    "Date": "Fecha"
}
datos = datos.rename(columns=renombrar_columnas)

columnas_necesarias = ["Producto", "Tienda", "Ventas", "Precio"]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if faltantes:
    raise ValueError(
        f"Faltan estas columnas: {', '.join(faltantes)}. "
        "Utiliza el archivo correspondiente a la Parte II."
    )

print("\nPRIMERAS 10 FILAS")
display(datos.head(10))


# --------------------------------------------------
# 2. PREPARAR LOS DATOS
# --------------------------------------------------

# Limpiar espacios y convertir textos vacíos en valores faltantes.
for columna in ["Producto", "Tienda"]:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

# Convertir ventas y precios a números.
for columna in ["Ventas", "Precio"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")
    datos[columna] = datos[columna].replace(
        [float("inf"), float("-inf")],
        float("nan")
    )

print("\nVALORES FALTANTES EN LAS COLUMNAS NECESARIAS")
print(datos[columnas_necesarias].isna().sum())

# No rellenamos ventas o precios con promedios porque inventaría ingresos.
filas_antes = len(datos)
datos = datos.dropna(subset=columnas_necesarias).copy()

print("Filas eliminadas por datos faltantes:", filas_antes - len(datos))

# Este análisis considera ventas de unidades enteras y no devoluciones.
filas_validas = (
    (datos["Ventas"] >= 0)
    & (datos["Ventas"] % 1 == 0)
    & (datos["Precio"] >= 0)
)

print("Filas con ventas o precios inválidos:", (~filas_validas).sum())
datos = datos.loc[filas_validas].copy()

if datos.empty:
    raise ValueError("No quedaron datos válidos para analizar.")

datos["Ventas"] = datos["Ventas"].astype(int)
datos = datos.reset_index(drop=True)

# No eliminamos filas iguales automáticamente:
# sin un identificador, podrían ser ventas distintas con los mismos valores.


# --------------------------------------------------
# 3. CREAR NUEVAS COLUMNAS
# --------------------------------------------------

# Calcular el importe de cada venta.
datos["Ingreso_Total"] = datos["Ventas"] * datos["Precio"]

# Normalizar las unidades vendidas a una escala entre 0 y 1.
venta_minima = datos["Ventas"].min()
venta_maxima = datos["Ventas"].max()

if venta_maxima > venta_minima:
    datos["Ventas_Normalizadas"] = (
        (datos["Ventas"] - venta_minima)
        / (venta_maxima - venta_minima)
    )
else:
    # Si todas las ventas son iguales, evitamos dividir por cero.
    datos["Ventas_Normalizadas"] = 0.0

# Clasificar las ventas según sus unidades.
# Límites de ejemplo:
# Baja: de 0 a 30.
# Media: más de 30 y hasta 50.
# Alta: más de 50.
# Estos límites se pueden ajustar según el negocio.
datos["Nivel_Venta"] = pd.cut(
    datos["Ventas"],
    bins=[-float("inf"), 30, 50, float("inf")],
    labels=["Baja", "Media", "Alta"]
)

# Si hay fechas, convertirlas y crear el período año-mes.
# Usar año y mes evita mezclar, por ejemplo, enero de distintos años.
if "Fecha" in datos.columns:
    datos["Fecha"] = pd.to_datetime(
        datos["Fecha"],
        errors="coerce"
    )

    datos["Mes"] = datos["Fecha"].dt.to_period("M")

    print("\nFechas faltantes o inválidas:", datos["Fecha"].isna().sum())
    print("Esas filas se conservarán, pero no entrarán en el resumen mensual.")

print("\nDATOS CON LAS NUEVAS COLUMNAS")
display(datos.head(10))


# --------------------------------------------------
# 4. AGRUPAR POR PRODUCTO Y TIENDA
# --------------------------------------------------

resumen_producto_tienda = datos.groupby(
    ["Producto", "Tienda"]
).agg(
    Ventas_Totales=("Ventas", "sum"),
    Venta_Promedio=("Ventas", "mean"),
    Cantidad_Registros=("Ventas", "count"),
    Venta_Minima=("Ventas", "min"),
    Venta_Maxima=("Ventas", "max"),
    Desviacion_Estandar=("Ventas", "std"),
    Varianza=("Ventas", "var"),
    Ingresos_Totales=("Ingreso_Total", "sum")
).reset_index()

resumen_producto_tienda = resumen_producto_tienda.sort_values(
    "Ingresos_Totales",
    ascending=False
)

print("\nRESUMEN POR PRODUCTO Y TIENDA")
display(resumen_producto_tienda.round(2))

# count cuenta registros, no unidades.
# std y var calculan estadísticas muestrales.
# Si un grupo tiene un solo registro, std y var serán NaN.


# --------------------------------------------------
# 5. AGRUPAR POR PRODUCTO Y MES
# --------------------------------------------------

if "Mes" in datos.columns:
    resumen_producto_mes = datos.dropna(subset=["Mes"]).groupby(
        ["Producto", "Mes"]
    ).agg(
        Ventas_Totales=("Ventas", "sum"),
        Venta_Promedio=("Ventas", "mean"),
        Cantidad_Registros=("Ventas", "count"),
        Venta_Minima=("Ventas", "min"),
        Venta_Maxima=("Ventas", "max"),
        Desviacion_Estandar=("Ventas", "std"),
        Varianza=("Ventas", "var"),
        Ingresos_Totales=("Ingreso_Total", "sum")
    ).reset_index()

    print("\nRESUMEN POR PRODUCTO Y MES")
    display(resumen_producto_mes.round(2))

else:
    print("\nNo se creó el resumen mensual porque el archivo no tiene Fecha.")


# --------------------------------------------------
# 6. ANÁLISIS PERSONALIZADO CON APPLY
# --------------------------------------------------

# transform devuelve la media del grupo en cada fila original.
# Así, cada venta queda accompagnée por la media de su producto y tienda.
datos["Media_Grupo"] = datos.groupby(
    ["Producto", "Tienda"]
)["Ventas"].transform("mean")


def analizar_venta(fila):
    # Diferencia en unidades respecto a la media del grupo.
    desviacion = fila["Ventas"] - fila["Media_Grupo"]

    # Diferencia porcentual respecto a esa misma media.
    # Si la media es cero, el porcentaje no está definido.
    if fila["Media_Grupo"] != 0:
        porcentaje = desviacion / fila["Media_Grupo"] * 100
    else:
        porcentaje = float("nan")

    # El margen evita clasificar pequeñas diferencias de redondeo.
    if abs(desviacion) < 0.000001:
        comparacion = "Igual a la media"
    elif desviacion > 0:
        comparacion = "Sobre la media"
    else:
        comparacion = "Bajo la media"

    return pd.Series({
        "Desviacion_Unidades": desviacion,
        "Desviacion_Porcentaje": porcentaje,
        "Comparacion_Grupo": comparacion
    })


# axis=1 aplica la función a cada fila.
# La función combina cálculos y condiciones personalizados.
resultados_personalizados = datos.apply(analizar_venta, axis=1)

datos[
    [
        "Desviacion_Unidades",
        "Desviacion_Porcentaje",
        "Comparacion_Grupo"
    ]
] = resultados_personalizados

print("\nVENTAS COMPARADAS CON LA MEDIA DE SU GRUPO")
display(
    datos[
        [
            "Producto",
            "Tienda",
            "Ventas",
            "Media_Grupo",
            "Desviacion_Unidades",
            "Desviacion_Porcentaje",
            "Comparacion_Grupo"
        ]
    ].head(10).round(2)
)

# Ejemplo de interpretación:
# Si una venta es 60 y la media del grupo es 40:
# desviación = 60 - 40 = 20 unidades.
# desviación porcentual = 20 / 40 × 100 = 50 %.


# --------------------------------------------------
# 7. RESUMIR LOS RESULTADOS
# --------------------------------------------------

print("\nRESUMEN POR NIVEL DE VENTA")

resumen_nivel = datos.groupby(
    "Nivel_Venta",
    observed=True
).agg(
    Cantidad_Registros=("Ventas", "count"),
    Ventas_Totales=("Ventas", "sum"),
    Ingresos_Totales=("Ingreso_Total", "sum")
)

display(resumen_nivel.round(2))

print("\nCOMPARACIÓN RESPECTO A LA MEDIA DEL GRUPO")
print(datos["Comparacion_Grupo"].value_counts())

# Mostrar todos los grupos empatados con el mayor ingreso.
mayor_ingreso = resumen_producto_tienda["Ingresos_Totales"].max()

mejores_grupos = resumen_producto_tienda[
    resumen_producto_tienda["Ingresos_Totales"] == mayor_ingreso
]

print("\nCOMBINACIONES DE PRODUCTO Y TIENDA CON MAYOR INGRESO")
display(
    mejores_grupos[
        ["Producto", "Tienda", "Ventas_Totales", "Ingresos_Totales"]
    ].round(2)
)

print("\nCONCLUSIONES GENERALES")
print("Registros analizados:", len(datos))
print("Productos diferentes:", datos["Producto"].nunique())
print("Tiendas diferentes:", datos["Tienda"].nunique())
print(f"Unidades vendidas: {datos['Ventas'].sum():,}")
print(f"Ingresos totales: {datos['Ingreso_Total'].sum():,.2f}")
print("Los ingresos no representan ganancias: no contamos con los costos.")


# --------------------------------------------------
# 8. GUARDAR LOS RESULTADOS
# --------------------------------------------------

datos.to_csv(
    "ventas_transformadas.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_producto_tienda.to_csv(
    "resumen_producto_tienda.csv",
    index=False,
    encoding="utf-8-sig"
)

if "Mes" in datos.columns:
    resumen_producto_mes.to_csv(
        "resumen_producto_mes.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("\nAnálisis finalizado.")
print("Los archivos generados están en el panel de archivos de Colab.")

Selecciona el archivo CSV utilizado en la Parte II.


Saving retail_sales_dataset.csv to retail_sales_dataset (1).csv

PRIMERAS 10 FILAS


,Transaction ID,Fecha,Tienda,Gender,Age,Producto,Ventas,Precio,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100
5,6,2023-04-25,CUST006,Female,45,Beauty,1,30,30
6,7,2023-03-13,CUST007,Male,46,Clothing,2,25,50
7,8,2023-02-22,CUST008,Male,30,Electronics,4,25,100
8,9,2023-12-13,CUST009,Male,63,Electronics,2,300,600
9,10,2023-10-07,CUST010,Female,52,Clothing,4,50,200



VALORES FALTANTES EN LAS COLUMNAS NECESARIAS
Producto    0
Tienda      0
Ventas      0
Precio      0
dtype: int64
Filas eliminadas por datos faltantes: 0
Filas con ventas o precios inválidos: 0

Fechas faltantes o inválidas: 0
Esas filas se conservarán, pero no entrarán en el resumen mensual.

DATOS CON LAS NUEVAS COLUMNAS


,Transaction ID,Fecha,Tienda,Gender,Age,Producto,Ventas,Precio,Total Amount,Ingreso_Total,Ventas_Normalizadas,Nivel_Venta,Mes
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,150,0.666667,Baja,2023-11
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,1000,0.333333,Baja,2023-02
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,30,0.000000,Baja,2023-01
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,500,0.000000,Baja,2023-05
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,100,0.333333,Baja,2023-05
5,6,2023-04-25,CUST006,Female,45,Beauty,1,30,30,30,0.000000,Baja,2023-04
6,7,2023-03-13,CUST007,Male,46,Clothing,2,25,50,50,0.333333,Baja,2023-03
7,8,2023-02-22,CUST008,Male,30,Electronics,4,25,100,100,1.000000,Baja,2023-02
8,9,2023-12-13,CUST009,Male,63,Electronics,2,300,600,600,0.333333,Baja,2023-12
9,10,2023-10-07,CUST010,Female,52,Clothing,4,50,200,200,1.000000,Baja,2023-10



RESUMEN POR PRODUCTO Y TIENDA


... (6 outputs left out)

,Cantidad_Registros,Ventas_Totales,Ingresos_Totales
Nivel_Venta,,,
Baja,1000,2514,456000



COMPARACIÓN RESPECTO A LA MEDIA DEL GRUPO
Comparacion_Grupo
Igual a la media    1000
Name: count, dtype: int64

COMBINACIONES DE PRODUCTO Y TIENDA CON MAYOR INGRESO


,Producto,Tienda,Ventas_Totales,Ingresos_Totales
979,Electronics,CUST927,4,2000
33,Beauty,CUST093,4,2000
474,Clothing,CUST476,4,2000
592,Clothing,CUST789,4,2000
515,Clothing,CUST595,4,2000
480,Clothing,CUST487,4,2000
46,Beauty,CUST139,4,2000
429,Clothing,CUST342,4,2000
508,Clothing,CUST572,4,2000
503,Clothing,CUST561,4,2000



CONCLUSIONES GENERALES
Registros analizados: 1000
Productos diferentes: 3
Tiendas diferentes: 1000
Unidades vendidas: 2,514
Ingresos totales: 456,000.00
Los ingresos no representan ganancias: no contamos con los costos.

Análisis finalizado.
Los archivos generados están en el panel de archivos de Colab.
